# Notebook 04 — Entrenamiento: Boceto Cartográfico → Satelital (B→A)

> **Blog Técnico** · Pix2Pix: Traducción `BtoA` — mapa de carreteras a fotografía aérea

Este notebook entrena el modelo en la dirección **inversa** respecto al notebook 03: dado un mapa de carreteras (OSM), el generador aprende a sintetizar la fotografía aérea equivalente.

## ¿Por qué la dirección importa?

Las dos direcciones de traducción son fundamentalmente asimétricas:

| Dirección | Entrada | Salida | Dificultad |
|-----------|---------|--------|------------|
| **AtoB** | Satelital (rico en textura) | Mapa (abstracto, líneas) | Media — el mapa tiene menos información |
| **BtoA** | Mapa (abstracto) | Satelital (textura compleja) | Alta — el modelo debe **alucinar** textura |

> **BtoA es más difícil** porque a partir de líneas abstractas hay que generar texturas fotorrealistas de edificios, vegetación y pavimento. Esto es una tarea de síntesis más que de abstracción.

In [ ]:
import sys, os
from pathlib import Path

PROYECTO = Path('..').resolve() if Path('../src').exists() else Path('.')
os.chdir(PROYECTO)
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

import torch
from train import config_desde_args, bucle_entrenamiento, generar_nombre_experimento

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1. Configuración para B→A

La única diferencia respecto al notebook 03 es `--direction BtoA`. El dataset es el mismo (`data/processed/`); el `DatasetParesSideBySide` simplemente invierte qué mitad es entrada y qué mitad es objetivo.

In [ ]:
ARGUMENTOS = [
    '--datos',           'data/processed',
    '--direction',       'BtoA',           # ← única diferencia respecto a notebook 03
    '--epocas',          '100',
    '--epocas_decay',    '100',
    '--batch',           '1',
    '--lr',              '0.0002',
    '--beta1',           '0.5',
    '--lambda_l1',       '100',
    '--gan_mode',        'lsgan',
    '--grad_accum',      '4',
    '--amp',
    '--frecuencia_ckpt', '10',
    '--frecuencia_muestra', '5',
    '--workers',         '2',
    # '--reanudar',
]

config, args = config_desde_args(ARGUMENTOS)
nombre_exp = generar_nombre_experimento(config, args)

config.imprimir_resumen()
print(f"  Nombre experimento : {nombre_exp}")

## 2. Lanzar Entrenamiento B→A

In [ ]:
entrenador = bucle_entrenamiento(
    config=config,
    nombre_exp=nombre_exp,
    args_extra=args,
)

print("\n[OK] Entrenamiento BtoA completado.")

## 3. Comparar A→B vs B→A

Una vez completados ambos entrenamientos, podemos comparar visualmente los resultados.

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image as PILImage

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparación de direcciones de traducción', fontsize=13)

for ax, exp_patron, titulo in [
    (axes[0], 'results/AtoB*/muestra_epoca_*.png', 'A→B (Satelital → Mapa)'),
    (axes[1], 'results/BtoA*/muestra_epoca_*.png', 'B→A (Mapa → Satelital)'),
]:
    archivos = sorted(glob.glob(exp_patron))
    if archivos:
        with PILImage.open(archivos[-1]) as img:
            ax.imshow(img)
        ax.set_title(titulo, fontsize=10)
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, 'Sin muestras todavía\n(entrenar primero)',
                ha='center', va='center', transform=ax.transAxes, fontsize=10)
        ax.set_title(titulo, fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.savefig('results/comparacion_direcciones.png', dpi=120, bbox_inches='tight')
plt.show()